# ASR Whisper Training on Google Colab A100
Run with Runtime → GPU A100.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


## Bootstrap (auto-detect Drive path)
`USE_LOCAL_SSD=1` creates a temporary Colab runtime copy for speed, not a permanent Drive duplicate.

This notebook now auto-detects `Colab_ASR_A100_Training` so it works even if the package is not exactly under `MyDrive/ASR_Colab_A100/`.


In [ ]:
import os, subprocess, pathlib, textwrap

os.environ['USE_LOCAL_SSD'] = '1'
os.environ.setdefault('A100_SYNC_INTERVAL_SEC', '600')

# Fast common paths first, then bounded find. Edit MANUAL_COLAB_ROOT if auto-detect fails.
MANUAL_COLAB_ROOT = ''  # e.g. '/content/drive/MyDrive/ASR_Colab_A100/Colab_ASR_A100_Training'
common_candidates = [
    MANUAL_COLAB_ROOT,
    '/content/drive/MyDrive/ASR_Colab_A100/Colab_ASR_A100_Training',
    '/content/drive/MyDrive/Colab_ASR_A100_Training',
]
found = []
for c in common_candidates:
    if c and pathlib.Path(c, 'scripts', 'colab_bootstrap_a100.sh').exists():
        found.append(c)

if not found:
    cmd = "find /content/drive/MyDrive /content/drive/Shareddrives -maxdepth 6 -type f -path '*/Colab_ASR_A100_Training/scripts/colab_bootstrap_a100.sh' 2>/dev/null | head -20"
    out = subprocess.getoutput(cmd).strip().splitlines()
    found = [str(pathlib.Path(x).parents[1]) for x in out if x.strip()]

if not found:
    raise FileNotFoundError('Cannot find Colab_ASR_A100_Training/scripts/colab_bootstrap_a100.sh in Google Drive. Upload the whole Colab_ASR_A100_Training folder or set MANUAL_COLAB_ROOT to its exact path.')

DRIVE_COLAB_ROOT = found[0]
DRIVE_PROJECT_ROOT = str(pathlib.Path(DRIVE_COLAB_ROOT).parent)
DRIVE_RESULTS_ROOT = str(pathlib.Path(DRIVE_PROJECT_ROOT) / 'Results')

os.environ['DRIVE_COLAB_ROOT'] = DRIVE_COLAB_ROOT
os.environ['DRIVE_PROJECT_ROOT'] = DRIVE_PROJECT_ROOT
os.environ['DRIVE_RESULTS_ROOT'] = DRIVE_RESULTS_ROOT

print('DRIVE_PROJECT_ROOT =', DRIVE_PROJECT_ROOT)
print('DRIVE_COLAB_ROOT   =', DRIVE_COLAB_ROOT)
print('DRIVE_RESULTS_ROOT =', DRIVE_RESULTS_ROOT)
print('USE_LOCAL_SSD      =', os.environ['USE_LOCAL_SSD'])
print('A100_SYNC_INTERVAL_SEC =', os.environ['A100_SYNC_INTERVAL_SEC'])

!bash "$DRIVE_COLAB_ROOT/scripts/colab_bootstrap_a100.sh"


## Train Whisper-small — paper-exact profile (recommended for paper)
Effective batch 32: batch 8 x grad_accum 4. This matches `RUN_GUIDE.md` and is the most defensible paper run.


In [ ]:
!bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_small_paper_exact.sh"


## Optional A100-fast Whisper-small
Use only if you explicitly choose speed over paper-exact microbatch parity. Effective batch remains 32, but microbatch/checkpointing differ.


In [ ]:
# !bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_small_a100_fast.sh"


## Optional Whisper-medium on A100


In [ ]:
# !bash "$DRIVE_COLAB_ROOT/scripts/colab_train_m02b_whisper_medium_a100.sh"


In [ ]:
!find "$DRIVE_RESULTS_ROOT" -maxdepth 5 -type f -name 'test_paper.json' -print
